In [20]:
import pandas as pd
import numpy as np

In [21]:
# Load dataset
road_df = pd.read_excel(
    "bitre_fatalities_jan2026.xlsx",
    sheet_name="BITRE_Fatality",
    skiprows=4
)

In [22]:
road_df.head()

,Crash ID,State,Month,Year,Dayweek,Time,Crash Type,Bus Involvement,Heavy Rigid Truck Involvement,Articulated Truck Involvement,Speed Limit,Road User,Gender,Age,National Remoteness Areas 2021,SA4 Name 2021,National LGA Name 2021,National Road Type,Christmas Period,Easter Period
0,1199112200569,NSW,12,1991,Friday,17:50:00,Single,No,-9,No,60,Pedestrian,Male,35,Unknown,Unknown,Unknown,Unknown,No,No
1,3199601280016,QLD,1,1996,Sunday,18:00:00,Single,No,-9,No,100,Passenger,Male,18,Unknown,Unknown,Unknown,Unknown,No,No
2,1370951612804693439,VIC,2,2025,Tuesday,09:45:00,Multiple,No,Yes,No,60,Driver,Female,82,Major Cities of Australia,Melbourne - North East,Whittlesea,Arterial Road,No,No
3,320181485224,QLD,8,2018,Sunday,11:00:00,Multiple,No,No,No,100,Motorcycle rider,Male,56,Outer Regional Australia,Wide Bay,Gympie,Local Road,No,No
4,1373485703321090654,QLD,10,2024,Sunday,16:00:00,Single,No,No,No,60,Driver,Male,56,Major Cities of Australia,Moreton Bay - North,Moreton Bay,Sub-arterial Road,No,No


In [23]:
road_df.columns

Index(['Crash ID', 'State', 'Month', 'Year', 'Dayweek', 'Time', 'Crash Type',
       'Bus Involvement', 'Heavy Rigid Truck Involvement',
       'Articulated Truck Involvement', 'Speed Limit', 'Road User', 'Gender',
       'Age', 'National Remoteness Areas 2021', 'SA4 Name 2021',
       'National LGA Name 2021', 'National Road Type', 'Christmas Period',
       'Easter Period'],
      dtype='object')

## BITRE Dataset Data Audit and cleaning

In [24]:
import pandas as pd
from pathlib import Path

# Load BITRE raw data

bitre_raw = pd.read_excel(
    "bitre_fatalities_jan2026.xlsx",
    sheet_name="BITRE_Fatality",
    skiprows=4
)

print("BITRE RAW DATA AUDIT")
print("=" * 80)

# Basic structure
print("Rows:", bitre_raw.shape[0])
print("Columns:", bitre_raw.shape[1])

print("\nCOLUMN LIST")
for i, col in enumerate(bitre_raw.columns, start=1):
    print(f"{i:02d}. {col}")

# Date coverage
print("\nDATE / TIME COVERAGE")
print("Year min:", bitre_raw["Year"].min())
print("Year max:", bitre_raw["Year"].max())
print("Month min:", bitre_raw["Month"].min())
print("Month max:", bitre_raw["Month"].max())

recent_2024_onwards = bitre_raw[bitre_raw["Year"] >= 2024]
recent_full_years = bitre_raw[bitre_raw["Year"].isin([2024, 2025])]

print("Records from 2024 onwards:", len(recent_2024_onwards))
print("Records in full recent years 2024-2025:", len(recent_full_years))

print("\nFatalities by year from 2020 onwards:")
print(
    bitre_raw[bitre_raw["Year"] >= 2020]["Year"]
    .value_counts()
    .sort_index()
)

# Duplicate checks
print("\nDUPLICATES")
print("Full duplicate rows:", bitre_raw.duplicated().sum())
print("Duplicate Crash ID rows:", bitre_raw["Crash ID"].duplicated().sum())
print("Unique Crash IDs:", bitre_raw["Crash ID"].nunique())
print("Note: duplicate Crash IDs can be valid because one crash can have multiple fatalities.")

# Standard missing values
print("\nSTANDARD MISSING VALUES")
missing = bitre_raw.isna().sum()
print(missing[missing > 0] if (missing > 0).any() else "No standard NaN values found.")

# Unknown and -9 audit
print("\nUNKNOWN / -9 VALUE AUDIT")
for col in bitre_raw.columns:
    col_text = bitre_raw[col].astype(str).str.strip()

    minus_9_count = (col_text == "-9").sum()
    unknown_count = (col_text.str.lower() == "unknown").sum()
    total_problem_values = minus_9_count + unknown_count

    if total_problem_values > 0:
        pct = total_problem_values / len(bitre_raw) * 100
        print(
            f"{col}: -9={minus_9_count:,}, "
            f"Unknown={unknown_count:,}, "
            f"Total={total_problem_values:,} ({pct:.1f}%)"
        )

# Recent unknown and -9 audit
print("\nRECENT UNKNOWN / -9 VALUE AUDIT: 2024 ONWARDS")
for col in recent_2024_onwards.columns:
    col_text = recent_2024_onwards[col].astype(str).str.strip()

    minus_9_count = (col_text == "-9").sum()
    unknown_count = (col_text.str.lower() == "unknown").sum()
    total_problem_values = minus_9_count + unknown_count

    if total_problem_values > 0:
        pct = total_problem_values / len(recent_2024_onwards) * 100
        print(
            f"{col}: -9={minus_9_count:,}, "
            f"Unknown={unknown_count:,}, "
            f"Total={total_problem_values:,} ({pct:.1f}%)"
        )

# Strange value checks
print("\nSTRANGE VALUE CHECKS")

print("State values:")
print(sorted(bitre_raw["State"].dropna().astype(str).unique()))

print("Month outside 1-12:")
print(((bitre_raw["Month"] < 1) | (bitre_raw["Month"] > 12)).sum())

print("Year outside 1989-2026:")
print(((bitre_raw["Year"] < 1989) | (bitre_raw["Year"] > 2026)).sum())

age_numeric = pd.to_numeric(bitre_raw["Age"], errors="coerce")
print("Age min:", age_numeric.min())
print("Age max:", age_numeric.max())
print("Age = -9:", (age_numeric == -9).sum())
print("Age > 100:", (age_numeric > 100).sum())

speed_numeric = pd.to_numeric(bitre_raw["Speed Limit"], errors="coerce")
print("Speed limit min excluding -9:", speed_numeric[speed_numeric != -9].min())
print("Speed limit max:", speed_numeric.max())
print("Speed limit = -9:", (speed_numeric == -9).sum())

# Key category counts
print("\nCATEGORY COUNTS")

category_cols = [
    "State",
    "Dayweek",
    "Crash Type",
    "Road User",
    "Gender",
    "National Remoteness Areas 2021",
    "National Road Type",
    "Christmas Period",
    "Easter Period"
]

for col in category_cols:
    print(f"\n{col}")
    print(
        bitre_raw[col]
        .astype(str)
        .str.strip()
        .value_counts()
        .head(20)
    )

print("\nRAW AUDIT CONCLUSION")
print("- The dataset is row-level fatality data: each row is one death.")
print("- It has time, person, place, and road-environment fields for narrative analysis.")
print("- Historical remoteness and road-type fields contain many Unknown values.")
print("- From 2024 onwards, those fields are much more complete.")
print("- Crash ID duplicates should not be removed automatically because one crash may involve multiple fatalities.")


BITRE RAW DATA AUDIT
Rows: 58284
Columns: 20

COLUMN LIST
01. Crash ID
02. State
03. Month
04. Year
05. Dayweek
06. Time
07. Crash Type
08. Bus Involvement
09. Heavy Rigid Truck Involvement
10. Articulated Truck Involvement
11. Speed Limit
12. Road User
13. Gender
14. Age
15. National Remoteness Areas 2021
16. SA4 Name 2021
17. National LGA Name 2021
18. National Road Type
19. Christmas Period
20. Easter Period

DATE / TIME COVERAGE
Year min: 1989
Year max: 2026
Month min: 1
Month max: 12
Records from 2024 onwards: 2714
Records in full recent years 2024-2025: 2608

Fatalities by year from 2020 onwards:
Year
2020    1097
2021    1129
2022    1182
2023    1251
2024    1292
2025    1316
2026     106
Name: count, dtype: int64

DUPLICATES
Full duplicate rows: 167
Duplicate Crash ID rows: 5685
Unique Crash IDs: 52599
Note: duplicate Crash IDs can be valid because one crash can have multiple fatalities.

STANDARD MISSING VALUES
No standard NaN values found.

UNKNOWN / -9 VALUE AUDIT
Crash Typ

Interpretation:
The BITRE dataset is suitable for a human-centred road safety narrative because each row represents one fatality and includes time, location, road-user, road-environment, and holiday-period variables. The data covers 58,284 fatalities from 1989 to January 2026, allowing long-term context as well as recent analysis.

The audit shows that standard blank values are not present, but missingness is encoded through -9 and “Unknown”. Historical records contain high unknown rates for remoteness, SA4, LGA, and road type fields, with around 78% unknown across the full dataset. However, the 2024 onwards records are much more complete, with only around 3-4% unknown for these fields. Therefore, the project will use the full historical dataset for trend context, while using 2024-Jan 2026 as the main analytical window for detailed dashboard storytelling.

Duplicate Crash IDs will not be removed automatically because one crash can involve multiple fatalities. Since each row represents a death, repeated Crash IDs can be valid and analytically meaningful.


In [25]:
import numpy as np
import pandas as pd
import re

# Make a copy so raw data stays untouched
bitre_clean = bitre_raw.copy()

# -----------------------------
# 1. Standardise column names
# -----------------------------
def clean_column_name(col):
    col = col.strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = col.strip("_")
    return col

bitre_clean.columns = [clean_column_name(col) for col in bitre_clean.columns]

print("Cleaned column names:")
print(bitre_clean.columns.tolist())


Cleaned column names:
['crash_id', 'state', 'month', 'year', 'dayweek', 'time', 'crash_type', 'bus_involvement', 'heavy_rigid_truck_involvement', 'articulated_truck_involvement', 'speed_limit', 'road_user', 'gender', 'age', 'national_remoteness_areas_2021', 'sa4_name_2021', 'national_lga_name_2021', 'national_road_type', 'christmas_period', 'easter_period']


In [26]:
# -----------------------------
# 2. Standardise missing values
# -----------------------------

# Replace numeric -9 and text "-9" with NaN first
bitre_clean = bitre_clean.replace(-9, np.nan)
bitre_clean = bitre_clean.replace("-9", np.nan)

# For categorical columns, fill missing values as "Unknown"
categorical_cols = [
    "state",
    "dayweek",
    "crash_type",
    "bus_involvement",
    "heavy_rigid_truck_involvement",
    "articulated_truck_involvement",
    "road_user",
    "gender",
    "national_remoteness_areas_2021",
    "sa4_name_2021",
    "national_lga_name_2021",
    "national_road_type",
    "christmas_period",
    "easter_period"
]

for col in categorical_cols:
    bitre_clean[col] = (
        bitre_clean[col]
        .astype("string")
        .str.strip()
        .fillna("Unknown")
    )

# Standardise state codes
bitre_clean["state"] = bitre_clean["state"].str.upper()

# Convert numeric columns
bitre_clean["year"] = pd.to_numeric(bitre_clean["year"], errors="coerce")
bitre_clean["month"] = pd.to_numeric(bitre_clean["month"], errors="coerce")
bitre_clean["age"] = pd.to_numeric(bitre_clean["age"], errors="coerce")
bitre_clean["speed_limit"] = pd.to_numeric(bitre_clean["speed_limit"], errors="coerce")

print("Missing values after standardisation:")
print(bitre_clean.isna().sum())


Missing values after standardisation:
crash_id                             0
state                                0
month                                0
year                                 0
dayweek                              0
time                                 0
crash_type                           0
bus_involvement                      0
heavy_rigid_truck_involvement        0
articulated_truck_involvement        0
speed_limit                       1485
road_user                            0
gender                               0
age                                110
national_remoteness_areas_2021       0
sa4_name_2021                        0
national_lga_name_2021               0
national_road_type                   0
christmas_period                     0
easter_period                        0
dtype: int64


In [27]:
# -----------------------------
# 3. Create month-level date
# -----------------------------

# BITRE gives year and month, but not exact crash day.
# So this is a month-start date for monthly analysis, not exact crash date.
bitre_clean["month_start"] = pd.to_datetime(
    {
        "year": bitre_clean["year"],
        "month": bitre_clean["month"],
        "day": 1
    },
    errors="coerce"
)

bitre_clean["year_month"] = bitre_clean["month_start"].dt.strftime("%Y-%m")
bitre_clean["month_name"] = bitre_clean["month_start"].dt.month_name()

# Each row represents one fatality
bitre_clean["deaths"] = 1

print(bitre_clean[["year", "month", "month_start", "year_month", "deaths"]].head())


   year  month month_start year_month  deaths
0  1991     12  1991-12-01    1991-12       1
1  1996      1  1996-01-01    1996-01       1
2  2025      2  2025-02-01    2025-02       1
3  2018      8  2018-08-01    2018-08       1
4  2024     10  2024-10-01    2024-10       1


In [28]:
# -----------------------------
# 4. Create age bands
# -----------------------------

def create_age_band(age):
    if pd.isna(age):
        return "Unknown"
    elif age <= 16:
        return "0-16"
    elif age <= 25:
        return "17-25"
    elif age <= 39:
        return "26-39"
    elif age <= 64:
        return "40-64"
    else:
        return "65+"

bitre_clean["age_band"] = bitre_clean["age"].apply(create_age_band)

print(bitre_clean["age_band"].value_counts())


age_band
40-64      15072
17-25      14784
26-39      13563
65+        10405
0-16        4350
Unknown      110
Name: count, dtype: int64


In [29]:
# -----------------------------
# 5. Create time bands
# -----------------------------

def extract_hour(value):
    if pd.isna(value):
        return np.nan

    # If already a Python time object
    if hasattr(value, "hour"):
        return value.hour

    text = str(value).strip()

    if text in ["", "Unknown", "nan"]:
        return np.nan

    match = re.match(r"^(\d{1,2})", text)
    if match:
        hour = int(match.group(1))
        if 0 <= hour <= 23:
            return hour

    return np.nan


def create_time_band(hour):
    if pd.isna(hour):
        return "Unknown"
    elif 0 <= hour <= 5:
        return "Late night 00-05"
    elif 6 <= hour <= 8:
        return "Morning commute 06-08"
    elif 9 <= hour <= 15:
        return "Daytime 09-15"
    elif 16 <= hour <= 18:
        return "Evening commute 16-18"
    else:
        return "Night 19-23"

bitre_clean["hour"] = bitre_clean["time"].apply(extract_hour)
bitre_clean["time_band"] = bitre_clean["hour"].apply(create_time_band)

print(bitre_clean[["time", "hour", "time_band"]].head())
print(bitre_clean["time_band"].value_counts())


       time  hour              time_band
0  17:50:00  17.0  Evening commute 16-18
1  18:00:00  18.0  Evening commute 16-18
2  09:45:00   9.0          Daytime 09-15
3  11:00:00  11.0          Daytime 09-15
4  16:00:00  16.0  Evening commute 16-18
time_band
Daytime 09-15            20232
Night 19-23              12034
Evening commute 16-18    10413
Late night 00-05          9506
Morning commute 06-08     6058
Unknown                     41
Name: count, dtype: int64


In [30]:
# -----------------------------
# 6. Create speed bands
# -----------------------------

def create_speed_band(speed):
    if pd.isna(speed):
        return "Unknown"
    elif speed <= 50:
        return "Local / urban <=50"
    elif speed <= 80:
        return "Urban arterial 60-80"
    elif speed <= 100:
        return "High speed 90-100"
    else:
        return "Very high speed 110+"

bitre_clean["speed_band"] = bitre_clean["speed_limit"].apply(create_speed_band)

print(bitre_clean["speed_band"].value_counts())


speed_band
Urban arterial 60-80    24803
High speed 90-100       21102
Very high speed 110+     6845
Local / urban <=50       4049
Unknown                  1485
Name: count, dtype: int64


In [31]:
# -----------------------------
# 7. Create road-user group
# -----------------------------

vulnerable_users = [
    "Pedestrian",
    "Pedal cyclist",
    "Motorcycle rider",
    "Motorcycle pillion passenger"
]

def create_road_user_group(user):
    if pd.isna(user) or user == "Unknown":
        return "Unknown"
    elif user in vulnerable_users:
        return "Vulnerable road user"
    else:
        return "Vehicle occupant"

bitre_clean["road_user_group"] = bitre_clean["road_user"].apply(create_road_user_group)

print(bitre_clean[["road_user", "road_user_group"]].drop_duplicates())


                        road_user       road_user_group
0                      Pedestrian  Vulnerable road user
1                       Passenger      Vehicle occupant
2                          Driver      Vehicle occupant
3                Motorcycle rider  Vulnerable road user
19                  Pedal cyclist  Vulnerable road user
24                        Unknown               Unknown
159  Motorcycle pillion passenger  Vulnerable road user


In [40]:
# -----------------------------
# 8. Create remoteness group
# -----------------------------

def create_remoteness_group(value):
    if pd.isna(value) or value == "Unknown":
        return "Unknown"
    elif value in ["Remote Australia", "Very Remote Australia"]:
        return "Remote / very remote"
    elif value == "Major Cities of Australia":
        return "Major Cities"
    elif value == "Inner Regional Australia":
        return "Inner Regional"
    elif value == "Outer Regional Australia":
        return "Outer Regional"
    else:
        return value

bitre_clean["remoteness_group"] = bitre_clean["national_remoteness_areas_2021"].apply(
    create_remoteness_group
)

print(bitre_clean["remoteness_group"].value_counts())



remoteness_group
Unknown                 45259
Major Cities             4614
Inner Regional           4261
Outer Regional           2942
Remote / very remote     1208
Name: count, dtype: int64


In [33]:
# -----------------------------
# 9. Create holiday period flag
# -----------------------------

bitre_clean["christmas_period"] = bitre_clean["christmas_period"].str.title()
bitre_clean["easter_period"] = bitre_clean["easter_period"].str.title()

bitre_clean["holiday_period"] = np.where(
    (bitre_clean["christmas_period"] == "Yes") |
    (bitre_clean["easter_period"] == "Yes"),
    "Christmas / Easter",
    "Non-holiday"
)

print(bitre_clean["holiday_period"].value_counts())


holiday_period
Non-holiday           56104
Christmas / Easter     2180
Name: count, dtype: int64


In [34]:
# -----------------------------
# 10. Create heavy vehicle flag
# -----------------------------

for col in [
    "bus_involvement",
    "heavy_rigid_truck_involvement",
    "articulated_truck_involvement"
]:
    bitre_clean[col] = bitre_clean[col].str.title()

def create_heavy_vehicle_flag(row):
    if (
        row["bus_involvement"] == "Yes" or
        row["heavy_rigid_truck_involvement"] == "Yes" or
        row["articulated_truck_involvement"] == "Yes"
    ):
        return "Yes"
    elif (
        row["bus_involvement"] == "No" and
        row["heavy_rigid_truck_involvement"] == "No" and
        row["articulated_truck_involvement"] == "No"
    ):
        return "No"
    else:
        return "Unknown"

bitre_clean["heavy_vehicle_involved"] = bitre_clean.apply(
    create_heavy_vehicle_flag,
    axis=1
)

print(bitre_clean["heavy_vehicle_involved"].value_counts())


heavy_vehicle_involved
No         31669
Unknown    17992
Yes         8623
Name: count, dtype: int64


In [35]:
# -----------------------------
# 11. Add analysis-window flags
# -----------------------------

bitre_clean["analysis_window"] = np.where(
    bitre_clean["year"] >= 2024,
    "Recent: 2024-Jan 2026",
    "Historical: 1989-2023"
)

bitre_clean["full_recent_year"] = np.where(
    bitre_clean["year"].isin([2024, 2025]),
    "2024-2025 full years",
    "Other"
)

print(bitre_clean["analysis_window"].value_counts())
print(bitre_clean["full_recent_year"].value_counts())


analysis_window
Historical: 1989-2023    55570
Recent: 2024-Jan 2026     2714
Name: count, dtype: int64
full_recent_year
Other                   55676
2024-2025 full years     2608
Name: count, dtype: int64


In [36]:
# -----------------------------
# 12. Final cleaning validation
# -----------------------------

print("Cleaned BITRE shape:", bitre_clean.shape)

print("\nRemaining missing values:")
print(bitre_clean.isna().sum()[bitre_clean.isna().sum() > 0])

print("\nYear range:")
print(bitre_clean["year"].min(), "to", bitre_clean["year"].max())

print("\nStates:")
print(sorted(bitre_clean["state"].unique()))

print("\nRecent records 2024 onwards:")
print(len(bitre_clean[bitre_clean["year"] >= 2024]))


Cleaned BITRE shape: (58284, 34)

Remaining missing values:
speed_limit    1485
age             110
hour             41
dtype: int64

Year range:
1989 to 2026

States:
['ACT', 'NSW', 'NT', 'QLD', 'SA', 'TAS', 'VIC', 'WA']

Recent records 2024 onwards:
2714


In [43]:
# Re-apply correct remoteness grouping
def create_remoteness_group(value):
    if pd.isna(value) or value == "Unknown":
        return "Unknown"
    elif value in ["Remote Australia", "Very Remote Australia"]:
        return "Remote / very remote"
    elif value == "Major Cities of Australia":
        return "Major Cities"
    elif value == "Inner Regional Australia":
        return "Inner Regional"
    elif value == "Outer Regional Australia":
        return "Outer Regional"
    else:
        return value

bitre_clean["remoteness_group"] = bitre_clean["national_remoteness_areas_2021"].apply(
    create_remoteness_group
)

# Recreate dependent datasets after fixing remoteness
bitre_recent = bitre_clean[bitre_clean["year"] >= 2024].copy()

bitre_recent_full_years = bitre_clean[
    bitre_clean["year"].isin([2024, 2025])
].copy()

bitre_state_month = (
    bitre_clean
    .groupby(["month_start", "year", "month", "year_month", "state"], as_index=False)
    .agg(fatalities=("deaths", "sum"))
)

# Validate again
print("BITRE clean detail:", bitre_clean.shape)
print("BITRE recent:", bitre_recent.shape)
print("BITRE recent full years:", bitre_recent_full_years.shape)
print("BITRE state month:", bitre_state_month.shape)

print("\nRecent remoteness groups:")
print(bitre_recent["remoteness_group"].value_counts())

BITRE clean detail: (58284, 34)
BITRE recent: (2714, 34)
BITRE recent full years: (2608, 34)
BITRE state month: (3367, 6)

Recent remoteness groups:
remoteness_group
Major Cities            968
Inner Regional          856
Outer Regional          592
Remote / very remote    218
Unknown                  80
Name: count, dtype: int64


In [44]:
# Create recent full-year dataset: 2024 and 2025 only
bitre_recent_full_years = bitre_clean[
    bitre_clean["year"].isin([2024, 2025])
].copy()

print("BITRE recent full years:", bitre_recent_full_years.shape)
print(bitre_recent_full_years["year"].value_counts().sort_index())


print("BITRE clean detail:", bitre_clean.shape)
print("BITRE recent:", bitre_recent.shape)
print("BITRE state month:", bitre_state_month.shape)

print("\nRecent year counts:")
print(bitre_recent["year"].value_counts().sort_index())

print("\nRecent remoteness groups:")
print(bitre_recent["remoteness_group"].value_counts())

print("\nRecent road user groups:")
print(bitre_recent["road_user_group"].value_counts())


BITRE recent full years: (2608, 34)
year
2024    1292
2025    1316
Name: count, dtype: int64
BITRE clean detail: (58284, 34)
BITRE recent: (2714, 34)
BITRE state month: (3367, 6)

Recent year counts:
year
2024    1292
2025    1316
2026     106
Name: count, dtype: int64

Recent remoteness groups:
remoteness_group
Major Cities            968
Inner Regional          856
Outer Regional          592
Remote / very remote    218
Unknown                  80
Name: count, dtype: int64

Recent road user groups:
road_user_group
Vehicle occupant        1619
Vulnerable road user    1051
Unknown                   44
Name: count, dtype: int64


In [45]:
# # -----------------------------
# # 13. Export cleaned BITRE files
# # -----------------------------

from pathlib import Path
from google.colab import files
import shutil

output_folder = Path("/content/clean_outputs")
output_folder.mkdir(parents=True, exist_ok=True)

bitre_clean.to_csv(output_folder / "bitre_clean_detail.csv", index=False)
bitre_recent.to_csv(output_folder / "bitre_clean_recent_2024_jan2026.csv", index=False)
bitre_recent_full_years.to_csv(
    output_folder / "bitre_clean_recent_full_years_2024_2025.csv",
    index=False
)
bitre_state_month.to_csv(output_folder / "bitre_state_month.csv", index=False)

bitre_state_month_recent = bitre_state_month[bitre_state_month["year"] >= 2024].copy()
bitre_state_month_recent.to_csv(
    output_folder / "bitre_state_month_recent_2024_jan2026.csv",
    index=False
)

print("Files exported:")
for file in output_folder.glob("*.csv"):
    print(file.name)

zip_path = shutil.make_archive(
    base_name="/content/bitre_clean_outputs",
    format="zip",
    root_dir=output_folder
)

files.download(zip_path)



Files exported:
bitre_state_month.csv
bitre_clean_recent_2024_jan2026.csv
bitre_clean_recent_full_years_2024_2025.csv
bitre_clean_detail.csv
bitre_state_month_recent_2024_jan2026.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Population Dataset Data Audit and Cleaning

In [46]:
# Load dataset
pop_df = pd.read_excel(
    "310104.xlsx",
    sheet_name="Data1",
    skiprows=4
)

In [48]:
import pandas as pd
import numpy as np

# Load population raw data
pop_raw = pd.read_excel(
    "310104.xlsx",
    sheet_name="Data1"
)

print("POPULATION RAW DATA AUDIT")
print("=" * 80)

print("Rows:", pop_raw.shape[0])
print("Columns:", pop_raw.shape[1])

print("\nFirst 12 values in first column:")
print(pop_raw.iloc[:12, 0].tolist())

print("\nColumn names:")
for i, col in enumerate(pop_raw.columns, start=1):
    print(f"{i:02d}. {col}")


POPULATION RAW DATA AUDIT
Rows: 186
Columns: 28

First 12 values in first column:
['Unit', 'Series Type', 'Data Type', 'Frequency', 'Collection Month', 'Series Start', 'Series End', 'No. Obs', 'Series ID', datetime.datetime(1981, 6, 1, 0, 0), datetime.datetime(1981, 9, 1, 0, 0), datetime.datetime(1981, 12, 1, 0, 0)]

Column names:
01. Unnamed: 0
02. Estimated Resident Population ;  Male ;  New South Wales ;
03. Estimated Resident Population ;  Male ;  Victoria ;
04. Estimated Resident Population ;  Male ;  Queensland ;
05. Estimated Resident Population ;  Male ;  South Australia ;
06. Estimated Resident Population ;  Male ;  Western Australia ;
07. Estimated Resident Population ;  Male ;  Tasmania ;
08. Estimated Resident Population ;  Male ;  Northern Territory ;
09. Estimated Resident Population ;  Male ;  Australian Capital Territory ;
10. Estimated Resident Population ;  Male ;  Australia ;
11. Estimated Resident Population ;  Female ;  New South Wales ;
12. Estimated Resident Popu

In [49]:
import pandas as pd

pop_raw = pd.read_excel("310104.xlsx", sheet_name="Data1")

print("POPULATION RAW DATA AUDIT")
print("=" * 80)

print("Rows:", pop_raw.shape[0])
print("Columns:", pop_raw.shape[1])

print("\nFirst 12 values in first column:")
print(pop_raw.iloc[:12, 0].tolist())

print("\nColumn groups:")
print("Male columns: B-J")
print("Female columns: K-S")
print("Persons columns: T-AB")

# Rename first column for audit
pop_audit = pop_raw.rename(columns={pop_raw.columns[0]: "date_or_metadata"})

# Parse actual date rows
pop_audit["parsed_date"] = pd.to_datetime(
    pop_audit["date_or_metadata"],
    errors="coerce"
)

date_rows = pop_audit[pop_audit["parsed_date"].notna()].copy()
metadata_rows = pop_audit[pop_audit["parsed_date"].isna()].copy()

print("\nMetadata rows:", len(metadata_rows))
print("Actual time-series rows:", len(date_rows))

print("\nDate coverage:")
print("Start:", date_rows["parsed_date"].min())
print("End:", date_rows["parsed_date"].max())

print("\nQuarter frequency check:")
print(date_rows["parsed_date"].dt.month.value_counts().sort_index())

# Identify Persons columns
person_cols = [
    col for col in pop_raw.columns
    if "Estimated Resident Population" in str(col)
    and "Persons" in str(col)
]

print("\nPersons population columns:", len(person_cols))
for col in person_cols:
    print("-", col)

print("\nMissing values in actual date rows:")
missing = date_rows.isna().sum()
print(missing[missing > 0] if (missing > 0).any() else "No missing values in date rows.")


POPULATION RAW DATA AUDIT
Rows: 186
Columns: 28

First 12 values in first column:
['Unit', 'Series Type', 'Data Type', 'Frequency', 'Collection Month', 'Series Start', 'Series End', 'No. Obs', 'Series ID', datetime.datetime(1981, 6, 1, 0, 0), datetime.datetime(1981, 9, 1, 0, 0), datetime.datetime(1981, 12, 1, 0, 0)]

Column groups:
Male columns: B-J
Female columns: K-S
Persons columns: T-AB

Metadata rows: 9
Actual time-series rows: 177

Date coverage:
Start: 1981-06-01 00:00:00
End: 2025-06-01 00:00:00

Quarter frequency check:
parsed_date
3     44
6     45
9     44
12    44
Name: count, dtype: int64

Persons population columns: 9
- Estimated Resident Population ;  Persons ;  New South Wales ;
- Estimated Resident Population ;  Persons ;  Victoria ;
- Estimated Resident Population ;  Persons ;  Queensland ;
- Estimated Resident Population ;  Persons ;  South Australia ;
- Estimated Resident Population ;  Persons ;  Western Australia ;
- Estimated Resident Population ;  Persons ;  Tasm

/tmp/ipykernel_3891/1202404502.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pop_audit["parsed_date"] = pd.to_datetime(


The ABS population workbook contains 177 quarterly observations from June 1981 to June 2025. The first nine rows are metadata, while the remaining rows contain quarterly population values. The file includes male, female, and persons population series by state and territory. For this project, the persons population fields are used as the denominator for fatality rates, because the dashboard compares overall road-fatality burden across states. No missing values were found in the actual time-series rows.


In [52]:
import pandas as pd

# -----------------------------
# POPULATION STRUCTURING
# Create tidy population table:
# year | quarter_date | state | state_name | population
# -----------------------------

# Load full ABS sheet without skiprows
pop_raw = pd.read_excel("310104.xlsx", sheet_name="Data1")

# Rename first column
pop_structured = pop_raw.rename(
    columns={pop_raw.columns[0]: "quarter_date"}
).copy()

# Convert first column to date
# Metadata rows become NaT
pop_structured["quarter_date"] = pd.to_datetime(
    pop_structured["quarter_date"],
    errors="coerce"
)

# Keep only real quarterly date rows
pop_structured = pop_structured[
    pop_structured["quarter_date"].notna()
].copy()

# Keep only total population/persons columns
person_cols = [
    col for col in pop_structured.columns
    if "Estimated Resident Population" in str(col)
    and "Persons" in str(col)
]

pop_structured = pop_structured[
    ["quarter_date"] + person_cols
].copy()

# Reshape from wide to long
pop_structured = pop_structured.melt(
    id_vars="quarter_date",
    var_name="series",
    value_name="population"
)

# Extract state name
pop_structured["state_name"] = pop_structured["series"].str.extract(
    r"Persons ;\s+(.+?)\s+;"
)

# Map state names to BITRE state codes
state_map = {
    "New South Wales": "NSW",
    "Victoria": "VIC",
    "Queensland": "QLD",
    "South Australia": "SA",
    "Western Australia": "WA",
    "Tasmania": "TAS",
    "Northern Territory": "NT",
    "Australian Capital Territory": "ACT",
    "Australia": "AUS"
}

pop_structured["state"] = pop_structured["state_name"].map(state_map)

# Add year field
pop_structured["year"] = pop_structured["quarter_date"].dt.year

# Convert population to numeric
pop_structured["population"] = pd.to_numeric(
    pop_structured["population"],
    errors="coerce"
)

# Keep final structured columns
pop_structured = pop_structured[
    ["year", "quarter_date", "state", "state_name", "population"]
].copy()

# Sort
pop_structured = pop_structured.sort_values(
    ["year", "state", "quarter_date"]
).reset_index(drop=True)

print("Structured population shape:", pop_structured.shape)
print(pop_structured.head(12))
print(pop_structured.tail(12))


Structured population shape: (1593, 5)
    year quarter_date state                    state_name  population
0   1981   1981-06-01   ACT  Australian Capital Territory      227581
1   1981   1981-09-01   ACT  Australian Capital Territory      228782
2   1981   1981-12-01   ACT  Australian Capital Territory      229484
3   1981   1981-06-01   AUS                     Australia    14923260
4   1981   1981-09-01   AUS                     Australia    14988677
5   1981   1981-12-01   AUS                     Australia    15054117
6   1981   1981-06-01   NSW               New South Wales     5234889
7   1981   1981-09-01   NSW               New South Wales     5249455
8   1981   1981-12-01   NSW               New South Wales     5266894
9   1981   1981-06-01    NT            Northern Territory      122616
10  1981   1981-09-01    NT            Northern Territory      125186
11  1981   1981-12-01    NT            Northern Territory      127718
      year quarter_date state          state_name  

/tmp/ipykernel_3891/2888132723.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pop_structured["quarter_date"] = pd.to_datetime(


In [54]:
# -----------------------------
# POPULATION CLEANING
# Create clean state-year population table
# -----------------------------

population_clean = pop_structured.copy()

# Remove Australia total because BITRE joins by state/territory
population_clean = population_clean[population_clean["state"] != "AUS"].copy()

# Keep June quarter only as annual population denominator
population_clean = population_clean[
    population_clean["quarter_date"].dt.month == 6
].copy()

# Ensure correct data types
population_clean["year"] = population_clean["year"].astype(int)
population_clean["state"] = population_clean["state"].astype(str)
population_clean["state_name"] = population_clean["state_name"].astype(str)
population_clean["population"] = pd.to_numeric(
    population_clean["population"],
    errors="coerce"
)

# Sort
population_clean = population_clean.sort_values(
    ["year", "state"]
).reset_index(drop=True)

print("Population clean shape:")
print(population_clean.shape)

print(population_clean.head())
print(population_clean.tail())


Population clean shape:
(360, 5)
   year quarter_date state                    state_name  population
0  1981   1981-06-01   ACT  Australian Capital Territory      227581
1  1981   1981-06-01   NSW               New South Wales     5234889
2  1981   1981-06-01    NT            Northern Territory      122616
3  1981   1981-06-01   QLD                    Queensland     2345208
4  1981   1981-06-01    SA               South Australia     1318769
     year quarter_date state         state_name  population
355  2025   2025-06-01   QLD         Queensland     5669834
356  2025   2025-06-01    SA    South Australia     1902331
357  2025   2025-06-01   TAS           Tasmania      575960
358  2025   2025-06-01   VIC           Victoria     7074468
359  2025   2025-06-01    WA  Western Australia     3043731


In [55]:
# -----------------------------
# POPULATION VALIDATION
# -----------------------------

print("Missing values:")
print(population_clean.isna().sum())

print("\nDuplicate state-year rows:")
print(population_clean.duplicated(subset=["year", "state"]).sum())

print("\nPopulation <= 0:")
print((population_clean["population"] <= 0).sum())

print("\nYear range:")
print(population_clean["year"].min(), "to", population_clean["year"].max())

print("\nStates:")
print(sorted(population_clean["state"].unique()))

print("\nRows per state:")
print(population_clean["state"].value_counts().sort_index())

print("\nNumber of states per year, latest 10 years:")
print(population_clean.groupby("year")["state"].nunique().tail(10))


Missing values:
year            0
quarter_date    0
state           0
state_name      0
population      0
dtype: int64

Duplicate state-year rows:
0

Population <= 0:
0

Year range:
1981 to 2025

States:
['ACT', 'NSW', 'NT', 'QLD', 'SA', 'TAS', 'VIC', 'WA']

Rows per state:
state
ACT    45
NSW    45
NT     45
QLD    45
SA     45
TAS    45
VIC    45
WA     45
Name: count, dtype: int64

Number of states per year, latest 10 years:
year
2016    8
2017    8
2018    8
2019    8
2020    8
2021    8
2022    8
2023    8
2024    8
2025    8
Name: state, dtype: int64


The ABS population data was cleaned into a state-year table using June-quarter population as the annual denominator. The Australia-wide total was removed because the project requires state-level joins with BITRE fatality data. The cleaned table contains 360 records across 8 states and territories from 1981 to 2025, with no missing values, no duplicate state-year records, and no invalid population values. This table is suitable for calculating fatalities per 100,000 residents by state and year.


In [56]:
from pathlib import Path
from google.colab import files

output_folder = Path("/content/clean_outputs")
output_folder.mkdir(parents=True, exist_ok=True)

population_clean.to_csv(
    output_folder / "population_state_year_clean.csv",
    index=False
)

files.download(output_folder / "population_state_year_clean.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Weather Dataset Data audit and data cleaning

In [ ]:
# ============================
# OPEN-METEO WEATHER EXTRACTION
# State -> Representative Capital City approach
# Colab-ready
# ============================

import requests
import pandas as pd
import time
from pathlib import Path
from google.colab import files

# --------------------------------
# 1. SETTINGS
# --------------------------------
START_DATE = "2024-01-01"
END_DATE = "2026-01-31"

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"

DAILY_VARS = [
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "rain_sum",
    "wind_speed_10m_max"
]

# --------------------------------
# 2. STATE -> REPRESENTATIVE CITY
# --------------------------------
location_map = {
    "NSW": {"city": "Sydney", "latitude": -33.8688, "longitude": 151.2093},
    "VIC": {"city": "Melbourne", "latitude": -37.8136, "longitude": 144.9631},
    "QLD": {"city": "Brisbane", "latitude": -27.4698, "longitude": 153.0251},
    "SA":  {"city": "Adelaide", "latitude": -34.9285, "longitude": 138.6007},
    "WA":  {"city": "Perth", "latitude": -31.9523, "longitude": 115.8613},
    "TAS": {"city": "Hobart", "latitude": -42.8821, "longitude": 147.3272},
    "NT":  {"city": "Darwin", "latitude": -12.4634, "longitude": 130.8456},
    "ACT": {"city": "Canberra", "latitude": -35.2809, "longitude": 149.1300},
}

# --------------------------------
# 3. FETCH DAILY WEATHER
# --------------------------------
def fetch_daily_weather(state, info):
    params = {
        "latitude": info["latitude"],
        "longitude": info["longitude"],
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": ",".join(DAILY_VARS),
        "timezone": "auto"
    }

    response = requests.get(ARCHIVE_URL, params=params, timeout=60)
    response.raise_for_status()

    data = response.json()
    daily = data["daily"]

    df = pd.DataFrame(daily)
    df["state"] = state
    df["city"] = info["city"]
    df["latitude"] = info["latitude"]
    df["longitude"] = info["longitude"]

    return df

frames = []
failed = []

for state, info in location_map.items():
    print(f"Fetching weather for {state} -> {info['city']}")

    try:
        df = fetch_daily_weather(state, info)
        frames.append(df)
        time.sleep(0.3)
    except Exception as e:
        failed.append((state, str(e)))
        print(f"Failed: {state} | {e}")

weather_daily = pd.concat(frames, ignore_index=True)

# --------------------------------
# 4. CLEAN DAILY WEATHER
# --------------------------------
weather_daily["date"] = pd.to_datetime(weather_daily["time"])
weather_daily["year"] = weather_daily["date"].dt.year
weather_daily["month"] = weather_daily["date"].dt.month
weather_daily["year_month"] = weather_daily["date"].dt.strftime("%Y-%m")

# Reorder columns
weather_daily = weather_daily[
    [
        "date",
        "year",
        "month",
        "year_month",
        "state",
        "city",
        "latitude",
        "longitude",
        "temperature_2m_mean",
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum",
        "rain_sum",
        "wind_speed_10m_max"
    ]
].copy()

# --------------------------------
# 5. MONTHLY AGGREGATION
# Join-ready for BITRE using state + year + month
# --------------------------------
weather_monthly = (
    weather_daily
    .groupby(["state", "city", "year", "month", "year_month"], as_index=False)
    .agg(
        avg_temp_mean=("temperature_2m_mean", "mean"),
        max_temp=("temperature_2m_max", "max"),
        min_temp=("temperature_2m_min", "min"),
        total_precipitation_mm=("precipitation_sum", "sum"),
        total_rain_mm=("rain_sum", "sum"),
        max_wind_speed_kmh=("wind_speed_10m_max", "max"),
        rainy_days=("rain_sum", lambda x: (x > 0).sum())
    )
)

# Add simple weather context flags
weather_monthly["rain_context"] = pd.cut(
    weather_monthly["total_rain_mm"],
    bins=[-1, 10, 50, 150, float("inf")],
    labels=["Low rain", "Moderate rain", "High rain", "Extreme rain"]
)

weather_monthly["heat_context"] = pd.cut(
    weather_monthly["max_temp"],
    bins=[-50, 30, 35, 40, float("inf")],
    labels=["Below 30C", "30-35C", "35-40C", "40C+"]
)

weather_monthly["wind_context"] = pd.cut(
    weather_monthly["max_wind_speed_kmh"],
    bins=[-1, 30, 50, 70, float("inf")],
    labels=["Low wind", "Moderate wind", "High wind", "Extreme wind"]
)

# --------------------------------
# 6. VALIDATION
# --------------------------------
print("\nWeather daily shape:", weather_daily.shape)
print("Weather monthly shape:", weather_monthly.shape)

print("\nDate range:")
print(weather_daily["date"].min(), "to", weather_daily["date"].max())

print("\nRows by state:")
print(weather_daily["state"].value_counts().sort_index())

print("\nMissing values:")
print(weather_daily.isna().sum())

if failed:
    print("\nFailed locations:")
    for state, err in failed:
        print(f"- {state}: {err}")
else:
    print("\nAll locations fetched successfully.")

# --------------------------------
# 7. SAVE OUTPUTS
# --------------------------------
output_folder = Path("/content/clean_outputs")
output_folder.mkdir(parents=True, exist_ok=True)

daily_path = output_folder / "weather_daily_state_2024_jan2026.csv"
monthly_path = output_folder / "weather_monthly_state_2024_jan2026.csv"

weather_daily.to_csv(daily_path, index=False)
weather_monthly.to_csv(monthly_path, index=False)

print("\nSaved files:")
print(daily_path)
print(monthly_path)

files.download(monthly_path)


In [57]:
# -----------------------------
# WEATHER RAW AUDIT
# -----------------------------

import pandas as pd

weather_raw = pd.read_csv("weather_monthly_state_2024_jan2026 (1).csv")

print("WEATHER RAW DATA AUDIT")
print("=" * 80)

print("Rows:", weather_raw.shape[0])
print("Columns:", weather_raw.shape[1])

print("\nColumn list:")
for i, col in enumerate(weather_raw.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nPreview:")
print(weather_raw.head())

print("\nDate coverage:")
print("Year-month min:", weather_raw["year_month"].min())
print("Year-month max:", weather_raw["year_month"].max())

print("\nYears and months:")
print(weather_raw.groupby("year")["month"].agg(["min", "max", "nunique"]))

print("\nStates:")
print(sorted(weather_raw["state"].unique()))

print("\nRows per state:")
print(weather_raw["state"].value_counts().sort_index())

print("\nMissing values:")
missing = weather_raw.isna().sum()
print(missing[missing > 0] if (missing > 0).any() else "No missing values")

print("\nDuplicate state-year-month rows:")
print(weather_raw.duplicated(subset=["state", "year", "month"]).sum())


WEATHER RAW DATA AUDIT
Rows: 200
Columns: 15

Column list:
01. state
02. city
03. year
04. month
05. year_month
06. avg_temp_mean
07. max_temp
08. min_temp
09. total_precipitation_mm
10. total_rain_mm
11. max_wind_speed_kmh
12. rainy_days
13. rain_context
14. heat_context
15. wind_context

Preview:
  state      city  year  month year_month  avg_temp_mean  max_temp  min_temp  \
0   ACT  Canberra  2024      1    2024-01      19.719355      30.2      10.6   
1   ACT  Canberra  2024      2    2024-02      19.841379      31.6       9.8   
2   ACT  Canberra  2024      3    2024-03      18.000000      32.2       5.7   
3   ACT  Canberra  2024      4    2024-04      12.246667      26.0       2.8   
4   ACT  Canberra  2024      5    2024-05       9.174194      16.8      -1.6   

   total_precipitation_mm  total_rain_mm  max_wind_speed_kmh  rainy_days  \
0                   105.0          105.0                26.9          20   
1                    51.4           51.4                23.0       

In [58]:
# -----------------------------
# WEATHER CLEANING / VALIDATION
# -----------------------------

weather_clean = weather_raw.copy()

# Standardise text fields
weather_clean["state"] = weather_clean["state"].astype(str).str.strip().str.upper()
weather_clean["city"] = weather_clean["city"].astype(str).str.strip()

# Ensure year/month are numeric
weather_clean["year"] = pd.to_numeric(
    weather_clean["year"],
    errors="coerce"
).astype("Int64")

weather_clean["month"] = pd.to_numeric(
    weather_clean["month"],
    errors="coerce"
).astype("Int64")

# Recreate year_month to guarantee consistency
weather_clean["year_month"] = (
    weather_clean["year"].astype(str) + "-" +
    weather_clean["month"].astype(str).str.zfill(2)
)

# Ensure weather measures are numeric
weather_numeric_cols = [
    "avg_temp_mean",
    "max_temp",
    "min_temp",
    "total_precipitation_mm",
    "total_rain_mm",
    "max_wind_speed_kmh",
    "rainy_days"
]

for col in weather_numeric_cols:
    weather_clean[col] = pd.to_numeric(
        weather_clean[col],
        errors="coerce"
    )

# Standardise context labels
context_cols = ["rain_context", "heat_context", "wind_context"]

for col in context_cols:
    weather_clean[col] = weather_clean[col].astype(str).str.strip()

# Sort
weather_clean = weather_clean.sort_values(
    ["state", "year", "month"]
).reset_index(drop=True)

print("Weather clean shape:")
print(weather_clean.shape)

print("\nMissing values after cleaning:")
missing = weather_clean.isna().sum()
print(missing[missing > 0] if (missing > 0).any() else "No missing values")

print("\nDuplicate state-year-month rows:")
print(weather_clean.duplicated(subset=["state", "year", "month"]).sum())

print("\nRows per state:")
print(weather_clean["state"].value_counts().sort_index())

print("\nNumeric ranges:")
for col in weather_numeric_cols:
    print(
        col,
        "min:", weather_clean[col].min(),
        "| max:", weather_clean[col].max()
    )


Weather clean shape:
(200, 15)

Missing values after cleaning:
No missing values

Duplicate state-year-month rows:
0

Rows per state:
state
ACT    25
NSW    25
NT     25
QLD    25
SA     25
TAS    25
VIC    25
WA     25
Name: count, dtype: int64

Numeric ranges:
avg_temp_mean min: 3.966666666666667 | max: 29.25483870967742
max_temp min: 13.9 | max: 44.3
min_temp min: -4.8 | max: 24.9
total_precipitation_mm min: 0.0 | max: 750.3
total_rain_mm min: 0.0 | max: 750.3
max_wind_speed_kmh min: 19.1 | max: 82.6
rainy_days min: 0 | max: 31


In [59]:
from pathlib import Path
from google.colab import files

output_folder = Path("/content/clean_outputs")
output_folder.mkdir(parents=True, exist_ok=True)

weather_clean.to_csv(
    output_folder / "weather_monthly_state_clean_2024_jan2026.csv",
    index=False
)

files.download(output_folder / "weather_monthly_state_clean_2024_jan2026.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Weather data was collected only for the latest analysis window, 2024-Jan 2026, because it is used as contextual enrichment for current policy decision-making rather than historical crash attribution.
